# Glunova Thermal Foot Diabetes Classification - Data Preparation & Augmentation

This notebook prepares data for a binary objective:
- `DF` (diabetic)
- `NO DF` (not diabetic)

It does three things:
1. Load raw ThermoFU images
2. Create leakage-safe stratified splits on raw families first
3. Apply offline augmentation per split with class balancing, then save to:
   - `/kaggle/working/train/class_name/`
   - `/kaggle/working/val/class_name/`
   - `/kaggle/working/test/class_name/`

The resulting `/kaggle/working` directory is ready to be published as Kaggle Dataset: **ThermoFU Preped**.

In [1]:
import random
import shutil
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

try:
    import albumentations as A
except Exception:
    !pip -q install albumentations
    import albumentations as A

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_ROOT = Path('/kaggle/input/datasets/yessinehakim1/thermofu/ThermoFU')
OUTPUT_ROOT = Path('/kaggle/working')
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}

BASE_AUG_PER_IMAGE = 2
TARGET_MODE = 'balance_to_max'  # options: balance_to_max, keep_original

print('INPUT_ROOT =', INPUT_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('BASE_AUG_PER_IMAGE =', BASE_AUG_PER_IMAGE)

INPUT_ROOT = /kaggle/input/datasets/yessinehakim1/thermofu/ThermoFU
OUTPUT_ROOT = /kaggle/working
BASE_AUG_PER_IMAGE = 2


In [2]:
def normalize_binary_label(raw_label: str) -> str:
    """Map different raw folder names to canonical DF labels."""
    s = str(raw_label).strip().upper().replace('-', ' ').replace('_', ' ')
    s = ' '.join(s.split())

    diabetic_aliases = {'DF', 'DM', 'DIABETIC', 'DIABETES', 'POSITIVE', 'YES'}
    non_diabetic_aliases = {'NO DF', 'NO DM', 'NON DIABETIC', 'NONDIABETIC', 'CONTROL', 'CG', 'NEGATIVE', 'NO'}

    if s in diabetic_aliases:
        return 'DF'
    if s in non_diabetic_aliases:
        return 'NO DF'
    raise ValueError(f'Unrecognized class folder name: {raw_label!r}')


def discover_raw_dataset(root: Path) -> pd.DataFrame:
    rows = []
    for p in root.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:
            continue
        rows.append({
            'path': str(p),
            'class_name': normalize_binary_label(p.parent.name),
        })

    if not rows:
        raise FileNotFoundError('No images found under /kaggle/input. Attach ThermoFU dataset first.')

    df = pd.DataFrame(rows)
    df = df[~df['path'].str.contains('/kaggle/working/', regex=False)].reset_index(drop=True)
    found_classes = set(df['class_name'].unique())
    expected = {'DF', 'NO DF'}
    if found_classes != expected:
        raise ValueError(f'Expected classes {expected}, found {found_classes}')
    return df


raw_df = discover_raw_dataset(INPUT_ROOT)
print('Class distribution BEFORE augmentation:')
print(raw_df['class_name'].value_counts().sort_index())

Class distribution BEFORE augmentation:
class_name
DF       244
NO DF     89
Name: count, dtype: int64


In [3]:
def build_aug_pipeline() -> A.Compose:
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.10),
        A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
        A.Affine(
            scale=(0.90, 1.12),
            translate_percent=(-0.06, 0.06),
            rotate=(-8, 8),
            shear=(-5, 5),
            mode=cv2.BORDER_REFLECT_101,
            p=0.6,
        ),
        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.6),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=12, val_shift_limit=12, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.25),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.25),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
    ])


def stratified_group_split_raw(raw_df: pd.DataFrame, seed: int = 42):
    fam_df = raw_df.copy().reset_index(drop=True)
    fam_df['family_id'] = fam_df.apply(lambda r: f"{r['class_name']}::{Path(r['path']).stem}", axis=1)

    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    idx_train, idx_temp = next(sss1.split(fam_df['family_id'], fam_df['class_name']))
    fam_train = fam_df.iloc[idx_train]
    fam_temp = fam_df.iloc[idx_temp]

    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    idx_val, idx_test = next(sss2.split(fam_temp['family_id'], fam_temp['class_name']))
    fam_val = fam_temp.iloc[idx_val]
    fam_test = fam_temp.iloc[idx_test]

    split_map = {}
    split_map.update({x: 'train' for x in fam_train['family_id']})
    split_map.update({x: 'val' for x in fam_val['family_id']})
    split_map.update({x: 'test' for x in fam_test['family_id']})

    out = fam_df.copy()
    out['split'] = out['family_id'].map(split_map)
    if out['split'].isna().any():
        raise RuntimeError('Some families were not assigned to a split.')
    assert out.groupby('family_id')['split'].nunique().max() == 1
    return out


raw_split_df = stratified_group_split_raw(raw_df, seed=SEED)
print('Raw class distribution AFTER split (before augmentation):')
display(pd.crosstab(raw_split_df['split'], raw_split_df['class_name']))

Raw class distribution AFTER split (before augmentation):


class_name,DF,NO DF
split,,
test,37,13
train,171,62
val,36,14


In [4]:
def make_augmented_index_per_split(
    raw_split_df: pd.DataFrame,
    base_aug_per_image: int = 2,
    mode: str = 'balance_to_max',
) -> pd.DataFrame:
    records = []

    for sp, split_part in raw_split_df.groupby('split'):
        split_counts = split_part['class_name'].value_counts()
        target_total = int(split_counts.max()) * (1 + base_aug_per_image) if mode == 'balance_to_max' else None

        for cls, df_cls in split_part.groupby('class_name'):
            df_cls = df_cls.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
            current_total = len(df_cls) * (1 + base_aug_per_image)
            extra_needed = max(0, target_total - current_total) if target_total is not None else 0

            extra_per = np.zeros(len(df_cls), dtype=int)
            if extra_needed > 0 and len(df_cls) > 0:
                order = np.arange(len(df_cls))
                np.random.shuffle(order)
                for i in range(extra_needed):
                    extra_per[order[i % len(df_cls)]] += 1

            for i, row in df_cls.iterrows():
                records.append({
                    'source_path': row['path'],
                    'class_name': cls,
                    'family_id': row['family_id'],
                    'split': sp,
                    'variant': 'orig',
                    'aug_idx': -1,
                })
                for k in range(base_aug_per_image):
                    records.append({
                        'source_path': row['path'],
                        'class_name': cls,
                        'family_id': row['family_id'],
                        'split': sp,
                        'variant': 'aug',
                        'aug_idx': k,
                    })
                for k in range(int(extra_per[i])):
                    records.append({
                        'source_path': row['path'],
                        'class_name': cls,
                        'family_id': row['family_id'],
                        'split': sp,
                        'variant': 'aug',
                        'aug_idx': base_aug_per_image + k,
                    })

    return pd.DataFrame(records)


split_df = make_augmented_index_per_split(raw_split_df, base_aug_per_image=BASE_AUG_PER_IMAGE, mode=TARGET_MODE)
print('Class distribution AFTER augmentation (balanced per split):')
display(pd.crosstab(split_df['split'], split_df['class_name']))

Class distribution AFTER augmentation (balanced per split):


class_name,DF,NO DF
split,,
test,111,111
train,513,513
val,108,108


In [5]:
def reset_output_dirs(output_root: Path, class_names):
    for sp in ['train', 'val', 'test']:
        split_dir = output_root / sp
        if split_dir.exists():
            shutil.rmtree(split_dir)
        for cls in class_names:
            (split_dir / cls).mkdir(parents=True, exist_ok=True)


def load_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Could not read image: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def save_rgb(path: Path, img_rgb: np.ndarray):
    ok = cv2.imwrite(str(path), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    if not ok:
        raise IOError(f'Failed writing image: {path}')


def export_split_dataset(split_df: pd.DataFrame, output_root: Path):
    class_names = sorted(split_df['class_name'].unique())
    reset_output_dirs(output_root, class_names)
    aug = build_aug_pipeline()
    counters = Counter()

    for _, row in split_df.iterrows():
        sp, cls, src = row['split'], row['class_name'], row['source_path']
        fam, variant, aug_idx = row['family_id'], row['variant'], int(row['aug_idx'])

        img = load_rgb(src)
        if variant == 'aug':
            img = aug(image=img)['image']
            name = f"{fam.replace('::', '__')}_aug_{aug_idx:03d}.png"
        else:
            name = f"{fam.replace('::', '__')}_orig.png"

        dst = output_root / sp / cls / name
        save_rgb(dst, img)
        counters[(sp, cls)] += 1

    return counters


final_counts = export_split_dataset(split_df, OUTPUT_ROOT)

print('Final saved counts per class per split:')
for sp in ['train', 'val', 'test']:
    print(f'[{sp}]')
    class_counts = []
    for cls in sorted(split_df['class_name'].unique()):
        cnt = final_counts[(sp, cls)]
        class_counts.append(cnt)
        print(f'  {cls}: {cnt}')
    if TARGET_MODE == 'balance_to_max':
        assert len(set(class_counts)) == 1, f'{sp} is not balanced: {class_counts}'

for sp in ['train', 'val', 'test']:
    n_aug = int((split_df['split'].eq(sp) & split_df['variant'].eq('aug')).sum())
    print(f'Augmented items in {sp}:', n_aug)
    assert n_aug > 0, f'No augmented samples in {sp}'

/tmp/ipykernel_55/4207415670.py:6: UserWarning: Argument(s) 'mode' are not valid for transform Affine
  A.Affine(


Final saved counts per class per split:
[train]
  DF: 513
  NO DF: 513
[val]
  DF: 108
  NO DF: 108
[test]
  DF: 111
  NO DF: 111
Augmented items in train: 793
Augmented items in val: 166
Augmented items in test: 172


## Output

Publish `/kaggle/working` as a Kaggle dataset named **ThermoFU Preped**.

Expected output structure:
- `/kaggle/working/train/<class_name>/*.png`
- `/kaggle/working/val/<class_name>/*.png`
- `/kaggle/working/test/<class_name>/*.png`

In [ ]:
# Create a downloadable ZIP containing train/val/test
zip_path = OUTPUT_ROOT / 'ThermoFU_Preped_splits.zip'

required_dirs = [OUTPUT_ROOT / 'train', OUTPUT_ROOT / 'val', OUTPUT_ROOT / 'test']
missing = [str(p) for p in required_dirs if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing split folders in /kaggle/working: {missing}')

if zip_path.exists():
    zip_path.unlink()

import zipfile

with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:
    for split_dir in required_dirs:
        for f in split_dir.rglob('*'):
            if f.is_file():
                # Keep paths inside ZIP as train/... , val/... , test/...
                zf.write(f, arcname=str(f.relative_to(OUTPUT_ROOT)))

size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'ZIP created: {zip_path}')
print(f'Size: {size_mb:.2f} MB')
print('Download it from the Kaggle Output files panel.')